In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/PROG74040-AI-Text-Detection"
)

print("Project exists:", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    print("\nProject contents:")
    for item in PROJECT_DIR.iterdir():
        print(" -", item.name)

Mounted at /content/drive
Project exists: True

Project contents:
 - data
 - models
 - outputs


In [3]:
MODELS_DIR = PROJECT_DIR / "models"

print("Models folder exists:", MODELS_DIR.exists())

if MODELS_DIR.exists():
    print("\nModels:")
    for item in MODELS_DIR.iterdir():
        print(" -", item.name)

Models folder exists: True

Models:
 - roberta_checkpoints
 - roberta_final
 - deberta_checkpoints
 - deberta_checkpoints_100k
 - deberta_final


In [4]:
DEBERTA_MODEL_DIR = MODELS_DIR / "deberta_final"

print("DeBERTa model exists:", DEBERTA_MODEL_DIR.exists())

DeBERTa model exists: True


In [5]:
CHECKPOINT_DIR = MODELS_DIR / "deberta_checkpoints_100k"

print("Checkpoint directory:", CHECKPOINT_DIR)

for item in CHECKPOINT_DIR.iterdir():
    print(" -", item.name)

Checkpoint directory: /content/drive/MyDrive/PROG74040-AI-Text-Detection/models/deberta_checkpoints_100k
 - checkpoint-6250


In [6]:
trainer.save_model(
    str(DEBERTA_FINAL_DIR)
)

NameError: name 'trainer' is not defined

In [7]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

DEBERTA_MODEL_DIR = MODELS_DIR / "deberta_final"

model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL_DIR
)

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-base"
)

print("DeBERTa model loaded successfully.")
print("Model path:", DEBERTA_MODEL_DIR)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

DeBERTa model loaded successfully.
Model path: /content/drive/MyDrive/PROG74040-AI-Text-Detection/models/deberta_final


In [8]:
from datasets import load_dataset

hc3 = load_dataset(
    "json",
    data_files="hf://datasets/Hello-SimpleAI/HC3/all.jsonl"
)

print(hc3)

all.jsonl: reconstructing file:   0%|          |  0.00B / 73.7MB            

all.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'human_answers', 'chatgpt_answers', 'index', 'source'],
        num_rows: 24322
    })
})


In [9]:
import pandas as pd

rows = []

for item in hc3["train"]:

    # Human = 0
    for answer in item["human_answers"]:
        if answer and answer.strip():
            rows.append({
                "text": answer.strip(),
                "label": 0
            })

    # AI = 1
    for answer in item["chatgpt_answers"]:
        if answer and answer.strip():
            rows.append({
                "text": answer.strip(),
                "label": 1
            })

hc3_df = pd.DataFrame(rows)

print("Before duplicate removal:", hc3_df.shape)
hc3_df.head()

Before duplicate removal: (85431, 2)


,text,label
0,"Basically there are many categories of "" Best ...",0
1,"If you 're hearing about it , it 's because it...",0
2,"One reason is lots of catagories . However , h...",0
3,There are many different best seller lists tha...,1
4,salt is good for not dying in car crashes and ...,0


In [10]:
hc3_df = hc3_df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

print("After duplicate removal:", hc3_df.shape)

print("\nClass distribution:")
print(hc3_df["label"].value_counts())

print("\nClass percentages:")
print(hc3_df["label"].value_counts(normalize=True) * 100)

After duplicate removal: (79331, 2)

Class distribution:
label
0    53086
1    26245
Name: count, dtype: int64

Class percentages:
label
0    66.917094
1    33.082906
Name: proportion, dtype: float64


In [11]:
from datasets import Dataset

hc3_dataset = Dataset.from_pandas(
    hc3_df,
    preserve_index=False
)

hc3_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 79331
})

In [12]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [13]:
tokenized_hc3 = hc3_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_hc3

Map:   0%|          | 0/79331 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 79331
})

In [14]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [15]:
from transformers import TrainingArguments

prediction_args = TrainingArguments(
    output_dir="/content/deberta_hc3_temp",

    per_device_eval_batch_size=16,

    report_to="none",

    fp16=False
)

In [16]:
from transformers import Trainer

hc3_trainer = Trainer(
    model=model,
    args=prediction_args
)

print("HC3 inference trainer created.")

HC3 inference trainer created.


In [17]:
pred_output = hc3_trainer.predict(
    tokenized_hc3
)

In [18]:
import numpy as np

logits = pred_output.predictions
labels = pred_output.label_ids

predictions = np.argmax(
    logits,
    axis=-1
)

probabilities = torch.softmax(
    torch.tensor(logits),
    dim=-1
)[:, 1].numpy()

print("Predictions:", len(predictions))
print("Labels:", len(labels))

Predictions: 79331
Labels: 79331


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

deberta_hc3_results = {
    "accuracy": accuracy_score(
        labels,
        predictions
    ),

    "precision": precision_score(
        labels,
        predictions,
        zero_division=0
    ),

    "recall": recall_score(
        labels,
        predictions,
        zero_division=0
    ),

    "f1": f1_score(
        labels,
        predictions,
        zero_division=0
    ),

    "roc_auc": roc_auc_score(
        labels,
        probabilities
    )
}

deberta_hc3_results

{'accuracy': 0.4985944964767871,
 'precision': 0.3975159042714329,
 'recall': 0.9999618975042865,
 'f1': 0.5688831084376524,
 'roc_auc': np.float64(0.9795499428179053)}

In [3]:
import pandas as pd

In [5]:
print("labels exists:", "labels" in globals())
print("predictions exists:", "predictions" in globals())
print("probabilities exists:", "probabilities" in globals())
print("OUTPUT_DIR exists:", "OUTPUT_DIR" in globals())

labels exists: False
predictions exists: False
probabilities exists: False
OUTPUT_DIR exists: False


In [6]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/PROG74040-AI-Text-Detection"
)

DEBERTA_MODEL_DIR = PROJECT_DIR / "models" / "deberta_final"
OUTPUT_DIR = PROJECT_DIR / "outputs"

print("Model exists:", DEBERTA_MODEL_DIR.exists())
print("Output exists:", OUTPUT_DIR.exists())

Mounted at /content/drive
Model exists: True
Output exists: True


In [7]:
import pandas as pd
import numpy as np
import torch

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-base"
)

model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL_DIR
)

print("DeBERTa loaded.")

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

DeBERTa loaded.


In [9]:
hc3 = load_dataset(
    "json",
    data_files="hf://datasets/Hello-SimpleAI/HC3/all.jsonl"
)

all.jsonl: reconstructing file:   0%|          |  0.00B / 73.7MB            

all.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [10]:
rows = []

for item in hc3["train"]:

    for answer in item["human_answers"]:
        if answer and answer.strip():
            rows.append({
                "text": answer.strip(),
                "label": 0
            })

    for answer in item["chatgpt_answers"]:
        if answer and answer.strip():
            rows.append({
                "text": answer.strip(),
                "label": 1
            })

hc3_df = pd.DataFrame(rows)

hc3_df = hc3_df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

print(hc3_df.shape)
print(hc3_df["label"].value_counts())

(79331, 2)
label
0    53086
1    26245
Name: count, dtype: int64


In [11]:
hc3_dataset = Dataset.from_pandas(
    hc3_df,
    preserve_index=False
)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_hc3 = hc3_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/79331 [00:00<?, ? examples/s]

In [12]:
prediction_args = TrainingArguments(
    output_dir="/content/deberta_hc3_temp",
    per_device_eval_batch_size=16,
    report_to="none",
    fp16=False
)

hc3_trainer = Trainer(
    model=model,
    args=prediction_args
)

In [13]:
pred_output = hc3_trainer.predict(
    tokenized_hc3
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
logits = pred_output.predictions
labels = pred_output.label_ids

predictions = np.argmax(
    logits,
    axis=-1
)

probabilities = torch.softmax(
    torch.tensor(logits),
    dim=-1
)[:, 1].numpy()

deberta_predictions_df = hc3_df.copy()

deberta_predictions_df["true_label"] = labels
deberta_predictions_df["prediction"] = predictions
deberta_predictions_df["ai_probability"] = probabilities

deberta_predictions_path = (
    OUTPUT_DIR / "hc3_deberta_predictions.csv"
)

deberta_predictions_df.to_csv(
    deberta_predictions_path,
    index=False
)

print("Saved to:")
print(deberta_predictions_path)

print("Rows:", len(deberta_predictions_df))

deberta_predictions_df.head()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

deberta_hc3_results = {
    "accuracy": accuracy_score(labels, predictions),
    "precision": precision_score(labels, predictions, zero_division=0),
    "recall": recall_score(labels, predictions, zero_division=0),
    "f1": f1_score(labels, predictions, zero_division=0),
    "roc_auc": roc_auc_score(labels, probabilities)
}

deberta_results_df = pd.DataFrame([{
    "model": "DeBERTa",
    "external_dataset": "HC3",
    **deberta_hc3_results
}])

deberta_results_df.to_csv(
    OUTPUT_DIR / "deberta_hc3_results.csv",
    index=False
)

deberta_hc3_results